# Naive Bayes Classifier

## 1. Brief Overview (in simple terms)

Naive Bayes is a **traditional supervised learning algorithm** used for **classification**. Like any supervised algorithm, it learns from labeled training data — pairs of (features, known class) — and uses what it learns to predict the class of new, unseen data points. It's based on probability. The idea is simple:

> Given the features (inputs) of a data point, what is the **most likely class (output)** it belongs to?

It uses **Bayes' Theorem** to flip the problem around. Instead of directly asking *"given these features, what's the class?"* (which is hard to know), it asks:

> *"If this were really class A, how likely am I to see these exact features? And how common is class A anyway?"*

It does this for every possible class and picks the one with the highest probability.

It's called **"Naive"** because it makes one big simplifying assumption: **all features are independent of each other**, given the class. In real life this is almost never fully true — e.g., in spam detection, the words "free" and "money" often appear together — but the algorithm still works surprisingly well in practice, especially for text classification (spam filters, sentiment analysis) because it's fast, simple, and needs very little training data.

**Its weakness:** the independence assumption is "naive" — if features are strongly correlated, the probability estimates can be off (though the final classification decision often still ends up correct).


## 2. The Mathematics

### 2.1 Bayes' Theorem — derivation

Bayes' theorem falls directly out of the definition of conditional probability. For two events $A$ and $B$, the conditional probability of $A$ given $B$ is defined as:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \qquad P(B) > 0$$

Similarly, the conditional probability of $B$ given $A$ is:

$$P(B \mid A) = \frac{P(A \cap B)}{P(A)}, \qquad P(A) > 0$$

Both expressions share the same numerator, $P(A \cap B)$. Rearranging each:

$$P(A \cap B) = P(A \mid B)\,P(B) \qquad \text{and} \qquad P(A \cap B) = P(B \mid A)\,P(A)$$

Since both equal $P(A \cap B)$, we can set them equal to each other:

$$P(A \mid B)\,P(B) = P(B \mid A)\,P(A)$$

Dividing both sides by $P(B)$ gives **Bayes' Theorem**:

$$\boxed{P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}}$$

This is a proof by construction — no assumption is needed beyond the basic definition of conditional probability, so it holds for *any* two events with non-zero probability.

### 2.2 Applying it to classification

Let $y$ be the class label (e.g. "spam" / "not spam") and $\mathbf{x} = (x_1, x_2, \dots, x_n)$ be the feature vector describing a data point. We want:

$$P(y \mid x_1, \dots, x_n)$$

i.e. "given these features, what's the probability of each class?" By Bayes' theorem:

$$P(y \mid x_1, \dots, x_n) = \frac{P(x_1, \dots, x_n \mid y)\, P(y)}{P(x_1, \dots, x_n)}$$

Where:
- $P(y)$ = **prior** — how common is this class overall (before seeing any features)?
- $P(x_1,\dots,x_n \mid y)$ = **likelihood** — how likely are these exact feature values, if the class really is $y$?
- $P(x_1,\dots,x_n)$ = **evidence** — probability of seeing this feature combination at all (same for every class, so it's just a normalizing constant)
- $P(y \mid x_1,\dots,x_n)$ = **posterior** — what we actually want: probability of the class given the features

> **Notation note:** writing several random variables separated by commas inside $P(\cdot)$, e.g. $P(x_1, x_2, \dots, x_n \mid y)$, means their **joint probability** — the probability that $x_1$ *and* $x_2$ *and* ... *and* $x_n$ all occur together, given $y$. It is shorthand for the intersection: $P(x_1, x_2, \dots, x_n \mid y) = P(x_1 \cap x_2 \cap \dots \cap x_n \mid y)$. This convention is used throughout — $P(A,B) = P(A \cap B)$.

### 2.3 The "Naive" independence assumption

The joint likelihood $P(x_1,\dots,x_n \mid y)$ is extremely hard to estimate directly — for $n$ features you'd need to model every possible interaction between them, requiring huge amounts of data.

Naive Bayes "cheats" by assuming every feature is **conditionally independent** of every other feature, given the class $y$. Recall for independent events $P(A \cap B) = P(A)P(B)$. Applying that assumption repeatedly:

$$P(x_1, x_2, \dots, x_n \mid y) = P(x_1 \mid y)\, P(x_2 \mid y) \cdots P(x_n \mid y) = \prod_{i=1}^{n} P(x_i \mid y)$$

Substituting this back into Bayes' theorem:

$$P(y \mid x_1, \dots, x_n) = \frac{P(y) \displaystyle\prod_{i=1}^{n} P(x_i \mid y)}{P(x_1, \dots, x_n)}$$

Since the denominator $P(x_1,\dots,x_n)$ is the same constant regardless of which class $y$ we plug in, it doesn't affect *which class wins* — we only need it to be proportional:

$$P(y \mid x_1, \dots, x_n) \;\propto\; P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

### 2.4 The decision rule (MAP estimate)

The classifier predicts the class $\hat{y}$ that **maximizes the posterior probability** — this is called the *Maximum A Posteriori* (MAP) rule:

$$\hat{y} = \arg\max_{y} \; P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

In practice, multiplying many small probabilities causes numerical underflow, so implementations (including scikit-learn) work in **log-space**, turning products into sums:

$$\hat{y} = \arg\max_{y} \; \left[ \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y) \right]$$

This is exactly equivalent mathematically (log is monotonically increasing, so it preserves the arg max) but numerically stable.

> **Why log is applied — underflow example:** suppose a document has $n=1000$ word-features, and each per-word likelihood $P(x_i \mid y)$ is a modest $0.01$. The raw product is
> $$\prod_{i=1}^{1000} P(x_i \mid y) \approx (0.01)^{1000} = 10^{-2000}$$
> That is far smaller than the smallest positive number a 64-bit float can represent (about $10^{-308}$). The running product **underflows to exactly $0.0$** — and once every class's score collapses to $0$, the classifier can no longer tell which class was actually most likely, even though the underlying math was correct.
>
> Taking logs turns the failing multiplication into a safe addition:
> $$\log \prod_{i=1}^{1000} P(x_i \mid y) = \sum_{i=1}^{1000} \log P(x_i \mid y) \approx 1000 \times \log(0.01) = -4600$$
> $-4600$ is an ordinary float value — no underflow. Since $\log$ is monotonically increasing, the class with the largest raw product still has the largest log-sum, so $\arg\max$ gives the identical decision either way — just computed on numbers that don't collapse to zero.

> **Notation note:** $\hat{y}$ ("y-hat") is standard notation for the model's *predicted* label, as opposed to the true label $y$. $\arg\max_y f(y)$ returns the *input* that maximizes $f$ — i.e. *which* class produced the largest score — not the score's value itself (that would just be $\max_y f(y)$).
>
> **$\max$ vs. $\arg\max$ example:** suppose we evaluate the score $f(y)$ for 3 candidate classes and get:
>
> | $y$ | $f(y)$ |
> |---|---|
> | class A | 0.02 |
> | class B | 0.15 |
> | class C | 0.08 |
>
> - $\max_y f(y) = 0.15$ — just the largest *value*
> - $\arg\max_y f(y) = \text{class B}$ — the *label* that produced that largest value
>
> A classifier needs the second one: it doesn't care how large the winning score is, only which class won.

### 2.5 Estimating the pieces from training data

- **Prior** $P(y)$: simply the fraction of training samples belonging to class $y$:
$$P(y) = \frac{\text{count}(y)}{N}$$

  where $\text{count}(y)$ is the number of training samples whose true label is $y$, and $N$ is the total number of training samples across *all* classes combined. It's just a frequency count — no probability theory needed, just counting rows. E.g. in the Wine demo below: if the training set has $N = 124$ samples total and 40 of them are cultivar `class_0`, then $\text{count}(\text{class\_0}) = 40$ and $P(\text{class\_0}) = 40/124 \approx 0.323$. This is exactly what `model.class_prior_` stores after `.fit()` — equivalent to `np.bincount(y_train) / len(y_train)`.

- **Likelihood** $P(x_i \mid y)$: depends on the *type* of feature, which is why there are different "flavors" of Naive Bayes:

  | Variant | Feature type | Likelihood model |
  |---|---|---|
  | **GaussianNB** | Continuous (real-valued) | Assumes $P(x_i \mid y)$ follows a Normal distribution: $\;P(x_i \mid y) = \dfrac{1}{\sqrt{2\pi\sigma_y^2}} \exp\!\left(-\dfrac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$, where $\mu_y, \sigma_y^2$ are the mean and variance of feature $i$ within class $y$ |
  | **MultinomialNB** | Discrete counts (e.g. word counts) | $P(x_i \mid y) = \dfrac{\text{count}(x_i, y) + \alpha}{\text{count}(y) + \alpha n}$ (with Laplace/additive smoothing $\alpha$ to avoid zero probabilities) |
  | **BernoulliNB** | Binary (0/1, e.g. word present/absent) | $P(x_i \mid y) = p_{i,y}^{x_i}(1-p_{i,y})^{1-x_i}$ |

  Smoothing (the $\alpha$ term) matters: without it, if a feature value never appears with a class in training data, its likelihood becomes exactly $0$, which zeroes out the *entire* product no matter how strong the other evidence is. Adding a small $\alpha$ (Laplace smoothing, $\alpha=1$ by default in sklearn) prevents this.

### 2.6 Worked mini-example

Suppose we classify an email as **Spam** or **Ham** using two binary features: contains "free" ($x_1$) and contains "win" ($x_2$).

Training data gives us: $P(\text{Spam}) = 0.4$, $P(\text{Ham}) = 0.6$, $P(\text{free}\mid \text{Spam})=0.7$, $P(\text{win}\mid\text{Spam})=0.6$, $P(\text{free}\mid\text{Ham})=0.1$, $P(\text{win}\mid\text{Ham})=0.05$.

For a new email containing **both** "free" and "win":

$$\text{score(Spam)} = P(\text{Spam}) \cdot P(\text{free}\mid\text{Spam}) \cdot P(\text{win}\mid\text{Spam}) = 0.4 \times 0.7 \times 0.6 = 0.168$$

$$\text{score(Ham)} = P(\text{Ham}) \cdot P(\text{free}\mid\text{Ham}) \cdot P(\text{win}\mid\text{Ham}) = 0.6 \times 0.1 \times 0.05 = 0.003$$

Since $0.168 \gg 0.003$, the email is classified as **Spam**. Normalizing (dividing each by their sum $0.171$) turns these scores into actual probabilities: $P(\text{Spam}\mid \mathbf{x}) \approx 98.2\%$.


### 2.7 The math behind each variant (simple version)

All three variants answer the exact same question — **"how likely is this one feature's value, if we're really looking at class $y$?"** — they just answer it differently depending on what *kind* of feature you have. Think of each one as a simple real-world analogy first, then look at the formula.

#### GaussianNB — "how close to average is this number?"

**Use when:** the feature is a plain number that can be anything (height, price, alcohol %).

**Analogy:** for each class, imagine you've drawn a bell curve using that class's training values — a peak at the class's *average*, spreading out based on how *varied* those values normally are. A new value's likelihood is just: **how tall is the bell curve at that point?** Close to the average → tall curve → high likelihood. Far from average → the curve has dropped off → low likelihood.

**The two numbers you need per class, per feature** (both are things you already know how to compute):
- $\mu$ (mu) = the **average** of that feature, using only that class's training rows
- $\sigma^2$ (sigma-squared) = the **variance** (how spread out the values are) of that feature, same rows

**The formula** just plugs those into the standard bell-curve shape:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x_i - \mu)^2}{2\sigma^2}\right)$$

Don't worry about memorizing this — the only thing to internalize is: **bigger gap between $x_i$ and $\mu$ → smaller probability.** Everything else in the formula just shapes the curve correctly so it's a valid probability distribution.

*Tiny example:* wine's average alcohol % for class A is $\mu=13.0$ with spread $\sigma^2=0.25$. A new wine with alcohol $13.1$ is very close to $13.0$ → high likelihood for class A. A wine with alcohol $15.0$ is far from $13.0$ → low likelihood.

#### MultinomialNB — "how often does this word show up in this class?"

**Use when:** the feature is a **count** (word counts in a document being the classic case).

**Analogy:** think of each class as a bag of words you draw from repeatedly. Some words are common in that bag (drawn often), some are rare. The likelihood of a word is just: **what fraction of the time does this word get pulled out of this class's bag?**

**The one number you need per class, per word:**

$$\theta_{i,y} = \frac{\text{how many times word } i \text{ appeared in class } y\text{'s training documents}}{\text{total words seen across all of class } y\text{'s training documents}}$$

That's it — just a percentage. If "free" makes up 5% of all words in Spam training emails, $\theta_{\text{free},\text{Spam}} = 0.05$.

*Tiny example:* out of 5000 total words across all Ham training emails, "meeting" appeared 150 times → $\theta_{\text{meeting},\text{Ham}} = 150/5000 = 0.03$ (3% of Ham's words are "meeting").

#### BernoulliNB — "does this word show up at all, yes or no?"

**Use when:** the feature is just **present or absent** (not how many times, just whether it appears).

**Analogy:** think of it like a biased coin per word, per class — flip it, and it tells you "yes this word shows up in a class-$y$ email" or "no it doesn't." The likelihood is just: **how often does that coin land "yes"?**

**The one number you need per class, per word:**

$$p_{i,y} = \frac{\text{number of class-}y\text{ documents that contain word } i \text{ at least once}}{\text{total number of class-}y\text{ documents}}$$

*Tiny example:* out of 100 Spam training emails, 60 of them contain the word "free" at least once (doesn't matter if once or five times) → $p_{\text{free},\text{Spam}} = 60/100 = 0.6$.

The formula just packages "yes" and "no" into one expression so code can compute it either way without an if/else:

$$P(x_i \mid y) = p_{i,y}^{\,x_i}\,(1-p_{i,y})^{\,1-x_i}$$

If the word is present ($x_i=1$) this evaluates to $p_{i,y}$; if absent ($x_i=0$) it evaluates to $1-p_{i,y}$.

#### The one-line summary

| Variant | Plain-English question it asks | What you compute |
|---|---|---|
| **GaussianNB** | How close is this number to the class's typical value? | average $\mu$ and spread $\sigma^2$ per feature, per class |
| **MultinomialNB** | What % of this class's words are this exact word? | count of word ÷ total words, per class |
| **BernoulliNB** | What % of this class's documents contain this word at all? | count of documents with word ÷ total documents, per class |

Every one of these numbers ($\mu$, $\sigma^2$, $\theta$, $p$) is called the **MLE** — expanded with the full calculus behind *why* averaging/counting is the mathematically "correct" answer in section 2.8, if you want to go deeper. For everyday understanding, it's enough to know: **all three variants boil down to counting or averaging your training data, split up by class.**

### 2.8 MLE and Laplace smoothing, in depth

#### What "Maximum Likelihood Estimation" actually means

Every formula in section 2.7 for $\mu_{i,y}$, $\theta_{i,y}$, and $p_{i,y}$ was labeled "the MLE." Here's what that means and where those formulas come from, not just what they are.

**The general recipe:** you assume the data was generated by some distribution with unknown parameter(s) $\theta$ (e.g. a Normal's mean/variance, or a coin's bias). The **likelihood function** $L(\theta)$ is the probability of the *observed* data, treated as a function of $\theta$:

$$L(\theta) = P(\text{observed data} \mid \theta)$$

MLE picks the $\theta$ that makes the data you actually saw as probable as possible:

$$\hat{\theta}_{\text{MLE}} = \arg\max_{\theta} L(\theta)$$

Just like in section 2.4, taking logs turns products into sums without changing which $\theta$ wins (log is monotonic), so in practice you maximize the **log-likelihood** $\ell(\theta) = \log L(\theta)$. To actually find the maximum, use ordinary calculus: take the derivative of $\ell(\theta)$ with respect to $\theta$, set it to $0$, and solve.

**Worked derivation — Gaussian mean $\mu$:** for $N_y$ training values $x^{(1)}, \dots, x^{(N_y)}$ of feature $i$ in class $y$, assumed i.i.d. $\mathcal{N}(\mu, \sigma^2)$:

$$\ell(\mu) = \sum_{k=1}^{N_y} \log\left[\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(x^{(k)}-\mu)^2}{2\sigma^2}\right)\right] = -\frac{N_y}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{k=1}^{N_y}(x^{(k)}-\mu)^2$$

Differentiate with respect to $\mu$ and set to $0$:

$$\frac{d\ell}{d\mu} = \frac{1}{\sigma^2}\sum_{k=1}^{N_y}\left(x^{(k)}-\mu\right) = 0 \;\;\Longrightarrow\;\; \sum_{k=1}^{N_y} x^{(k)} = N_y\,\mu \;\;\Longrightarrow\;\; \hat{\mu} = \frac{1}{N_y}\sum_{k=1}^{N_y} x^{(k)}$$

That last expression is exactly the ordinary **sample mean** — which is why $\mu_{i,y}$ in section 2.7 is "just the average." The same derivative trick applied to $\sigma^2$ (differentiating $\ell$ with respect to $\sigma^2$ instead) yields the sample variance $\hat{\sigma}^2 = \frac{1}{N_y}\sum_k (x^{(k)}-\hat\mu)^2$ — this is *why* GaussianNB's `theta_` and `var_` are simple averages rather than something more exotic: they are provably the values that make the training data most likely under a Gaussian assumption.

**Worked derivation — Multinomial $\theta_i$:** maximizing $\log\prod_i \theta_i^{x_i}$ subject to the constraint $\sum_i \theta_i = 1$ (probabilities must sum to 1) requires a Lagrange multiplier $\lambda$:

$$\mathcal{L} = \sum_i x_i \log\theta_i + \lambda\left(1 - \sum_i\theta_i\right)$$

$$\frac{\partial \mathcal{L}}{\partial \theta_i} = \frac{x_i}{\theta_i} - \lambda = 0 \;\;\Longrightarrow\;\; \theta_i = \frac{x_i}{\lambda}$$

Summing both sides over $i$ and using $\sum_i \theta_i = 1$ pins down $\lambda = \sum_i x_i$ (the total word count), giving $\hat\theta_i = x_i / \sum_i x_i$ — the relative frequency formula from section 2.7. The Bernoulli case follows the same pattern (differentiate $\sum_k[x_k\log p + (1-x_k)\log(1-p)]$, set to $0$) and also collapses to a simple relative frequency. **The pattern to notice:** every "just count and divide" formula you see in Naive Bayes is not an ad-hoc shortcut — it's the exact output of the MLE calculus, which happens to simplify to counting for these particular distributions.

#### Laplace smoothing, derived and worked numerically

**The zero-probability problem, concretely.** Suppose a spam filter's vocabulary has $n=1000$ words, and the word "lottery" happens to never appear in any Ham training email, while Ham class has $\text{count}(\text{Ham}) = 5000$ total word occurrences. The raw MLE gives:

$$\theta_{\text{lottery},\,\text{Ham}} = \frac{\text{count}(\text{lottery}, \text{Ham})}{\text{count}(\text{Ham})} = \frac{0}{5000} = 0$$

Now any test email containing "lottery" gets $P(\text{Ham} \mid \mathbf{x}) \propto P(\text{Ham}) \cdot \theta_{\text{lottery},\text{Ham}} \cdot (\text{everything else}) = 0$ — **exactly zero**, no matter how strongly every *other* word in that email points to Ham. One unseen word silently overrides all other evidence. This happens easily in practice: any word absent from a class's training data, even by chance with limited data, zeroes that class out completely.

**Where the fix comes from.** Laplace smoothing isn't just a patch — it falls out of treating $\theta_{i,y}$ not as a single fixed unknown to point-estimate, but as a random variable with a **prior belief** attached (a Bayesian, rather than purely frequentist, approach). Specifically, put a *uniform* prior over the possible values of $\theta_{i,y}$ before seeing any data (formally, a Dirichlet distribution with all parameters equal to 1, which encodes "I have no prior preference — pretend I already saw 1 occurrence of every word"). Combining this prior with the observed counts via Bayes' theorem and taking the resulting posterior estimate gives exactly:

$$\theta_{i,y} = \frac{\text{count}(x_i, y) + \alpha}{\text{count}(y) + \alpha n}$$

Each of the $n$ vocabulary words is credited with $\alpha$ "pretend" occurrences ($\alpha=1$ is the classic Laplace "add-one" rule) before the real counts are added in. That's also why the denominator adds $\alpha n$ rather than just $\alpha$: it must add back $\alpha$ for *every* one of the $n$ words to keep $\sum_i \theta_{i,y} = 1$.

**The same numeric example, smoothed** ($\alpha=1$, $n=1000$, $\text{count}(\text{lottery},\text{Ham})=0$, $\text{count}(\text{Ham})=5000$):

$$\theta_{\text{lottery},\,\text{Ham}} = \frac{0 + 1}{5000 + 1\times 1000} = \frac{1}{6000} \approx 0.000167$$

Still small — "lottery" is genuinely rare-to-absent in Ham — but **no longer zero**. A Ham-leaning email containing "lottery" now gets a small penalty instead of being unconditionally forced to Spam. This is exactly the `alpha` parameter passed to `MultinomialNB(alpha=1.0)` and `BernoulliNB(alpha=1.0)` in scikit-learn (both default to $\alpha=1$); setting `alpha` closer to $0$ approaches raw MLE (more zero-probability risk), while larger `alpha` smooths estimates more aggressively toward uniform (more bias, less variance).

### 2.9 A simple end-to-end example tying it all together

Classify a fruit as **Banana** or **Apple** using two yes/no features: is it **Yellow**, and is it **Long**-shaped.

**Training data (10 fruits: 6 Banana, 4 Apple):**

| | Yellow = 1 | Long = 1 |
|---|---|---|
| Banana (6 total) | 5 of them | 6 of them |
| Apple (4 total) | 1 of them | 0 of them |

**Step 1 — Prior $P(y)$** (section 2.5/2.8: just counting, the MLE):

$$P(\text{Banana}) = 6/10 = 0.6 \qquad P(\text{Apple}) = 4/10 = 0.4$$

**Step 2 — Likelihoods $P(x_i\mid y)$** (section 2.7: Bernoulli-style, also just counting):

$$P(\text{Yellow}\mid\text{Banana})=5/6\approx0.833 \qquad P(\text{Yellow}\mid\text{Apple})=1/4=0.25$$

$$P(\text{Long}\mid\text{Banana})=6/6=1.0 \qquad P(\text{Long}\mid\text{Apple})=0/4=\mathbf{0}$$

**New fruit to classify:** Yellow = 1, Long = 1. Using the independence assumption (section 2.3), multiply prior × likelihoods:

$$\text{score(Banana)} = 0.6\times0.833\times1.0 = 0.5 \qquad \text{score(Apple)} = 0.4\times0.25\times\mathbf{0} = \mathbf{0}$$

**The zero-probability problem (section 2.8) shows up immediately** — no Apple in training was ever Long, so Apple's score is forced to exactly zero. The model claims 100% certainty it's a Banana, which overstates its confidence (a long yellow apple is rare, not impossible).

**Step 3 — Laplace smoothing fixes it** ($\alpha=1$, 2 possible outcomes per feature so the denominator gets $+2\alpha$):

$$P(\text{Yellow}\mid\text{Banana})=\tfrac{5+1}{6+2}=0.75 \qquad P(\text{Yellow}\mid\text{Apple})=\tfrac{1+1}{4+2}\approx0.333$$

$$P(\text{Long}\mid\text{Banana})=\tfrac{6+1}{6+2}=0.875 \qquad P(\text{Long}\mid\text{Apple})=\tfrac{0+1}{4+2}\approx0.167$$

Recomputing: $\text{score(Banana)} = 0.6\times0.75\times0.875 = 0.394$, $\text{score(Apple)} = 0.4\times0.333\times0.167 \approx 0.022$. Apple is no longer impossible, just unlikely.

**Step 4 — log-space** (section 2.4, same decision, safe from underflow):

$$\log\text{score(Banana)} = \log0.6+\log0.75+\log0.875 \approx -0.932$$

$$\log\text{score(Apple)} = \log0.4+\log0.333+\log0.167 \approx -3.806$$


## 3. Demo using scikit-learn

We'll use `GaussianNB` (continuous features) on the classic **Wine** dataset — predicting which of 3 cultivars a wine came from, based on 13 chemical measurements (alcohol content, color intensity, flavanoids, etc.).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load dataset
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names
class_names = wine.target_names

df = pd.DataFrame(X, columns=feature_names)
df['cultivar'] = class_names[y]  # look up each sample's class name by its integer label in y
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,cultivar
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,class_0


In [ ]:
# Split into train/test sets (stratify=y keeps class proportions the same in both splits)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train the Naive Bayes classifier.
# .fit() does NOT do iterative optimization like gradient descent -- for GaussianNB
# it just computes, per class: the prior P(y), and the mean/variance of every
# feature restricted to that class's training rows (section 2.7's MLE formulas).
model = GaussianNB()
model.fit(X_train, y_train)

# class_prior_ = P(y) for each class -- count(y) / N, section 2.5
print("Class priors P(y):", dict(zip(class_names, model.class_prior_)))

# theta_[class, feature] = mu_{i,y}  -- the learned mean of each feature, per class
print("\nPer-class feature means (mu):\n", pd.DataFrame(model.theta_, index=class_names, columns=feature_names))

# var_[class, feature] = sigma^2_{i,y}  -- the learned variance of each feature, per class
# Together, theta_ and var_ fully define the per-class Gaussian curve used for P(x_i | y)
print("\nPer-class feature variances (sigma^2):\n", pd.DataFrame(model.var_, index=class_names, columns=feature_names))

Class priors P(y): {np.str_('class_0'): np.float64(0.33064516129032256), np.str_('class_1'): np.float64(0.4032258064516129), np.str_('class_2'): np.float64(0.2661290322580645)}

Per-class feature means (mu):
            alcohol  malic_acid       ash  alcalinity_of_ash   magnesium  \
class_0  13.730488    1.947073  2.449756          17.102439  106.634146   
class_1  12.242400    1.962600  2.232800          20.524000   95.140000   
class_2  13.074545    3.200909  2.454242          21.560606   99.272727   

         total_phenols  flavanoids  nonflavanoid_phenols  proanthocyanins  \
class_0       2.828537    2.940244              0.301707         1.851220   
class_1       2.253600    2.046800              0.350800         1.712200   
class_2       1.687576    0.787576              0.446364         1.138788   

         color_intensity       hue  od280/od315_of_diluted_wines      proline  
class_0         5.567805  1.050976                      3.088537  1112.804878  
class_1         2.960

In [ ]:
# Predict on the test set.
# Internally, predict() computes log P(y) + sum_i log P(x_i | y) for every class
# (the log-space MAP rule from section 2.4) and returns the argmax class per sample.
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
# Confusion matrix: rows = true class, columns = predicted class -- diagonal = correct predictions
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
# Precision/recall/F1 per class, to see if any single class is harder than the others
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=class_names))

Accuracy: 1.0

Confusion matrix:
 [[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]

Classification report:
               precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54



In [ ]:
# Inspect the posterior probabilities for a single sample -- this is the P(y | x) from the math above
sample = X_test[0].reshape(1, -1)  # reshape to a (1, n_features) 2D array, since predict expects a batch

# predict_proba() = the softmax of the per-class log-scores (section 2.4's softmax note):
# it exponentiates each class's log P(y) + sum_i log P(x_i|y), then normalizes so they sum to 1.
proba = model.predict_proba(sample)

print("Sample features:", dict(zip(feature_names, X_test[0])))
print("True label:", class_names[y_test[0]])
print("\nPosterior probabilities P(class | features):")
for cls, p in zip(class_names, proba[0]):
    print(f"  {cls:12s}: {p:.4f}")

# predict() = argmax over these posterior probabilities -- the single winning class
print("\nPredicted class:", class_names[model.predict(sample)[0]])

Sample features: {'alcohol': np.float64(13.16), 'malic_acid': np.float64(2.36), 'ash': np.float64(2.67), 'alcalinity_of_ash': np.float64(18.6), 'magnesium': np.float64(101.0), 'total_phenols': np.float64(2.8), 'flavanoids': np.float64(3.24), 'nonflavanoid_phenols': np.float64(0.3), 'proanthocyanins': np.float64(2.81), 'color_intensity': np.float64(5.68), 'hue': np.float64(1.03), 'od280/od315_of_diluted_wines': np.float64(3.17), 'proline': np.float64(1185.0)}
True label: class_0

Posterior probabilities P(class | features):
  class_0     : 1.0000
  class_1     : 0.0000
  class_2     : 0.0000

Predicted class: class_0


### Bonus: `MultinomialNB` for text classification

The most common real-world use of Naive Bayes is text classification (spam filtering, topic/sentiment classification), using word counts as features.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Tiny toy dataset: 1 = spam-like, 0 = ham-like
texts = [
    "win a free prize now",
    "free money click now to win",
    "meeting scheduled for tomorrow at noon",
    "please review the attached report",
    "you have won a free lottery, claim now",
    "let's catch up over lunch tomorrow",
]
labels = [1, 1, 0, 0, 1, 0]

# CountVectorizer builds the vocabulary and turns each text into a row of word counts --
# this is exactly the x_1..x_n count vector the multinomial model expects (section 2.7).
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(texts)  # bag-of-words counts, this is x_i

# alpha = the Laplace smoothing term added to every count(x_i, y) in section 2.7's
# theta_{i,y} formula, so a word unseen in a class doesn't zero out that class's whole score.
nb_text = MultinomialNB(alpha=1.0)
nb_text.fit(X_text, labels)  # fit() just counts word frequencies per class -- no optimization loop

new_msgs = ["free lunch meeting tomorrow", "win free money now"]
new_X = vectorizer.transform(new_msgs)  # reuse the SAME vocabulary learned from training text
preds = nb_text.predict(new_X)
probs = nb_text.predict_proba(new_X)

for msg, pred, prob in zip(new_msgs, preds, probs):
    label = "spam-like" if pred == 1 else "ham-like"
    print(f"'{msg}' -> {label}  (P(spam)={prob[1]:.3f})")

'free lunch meeting tomorrow' -> ham-like  (P(spam)=0.250)
'win free money now' -> spam-like  (P(spam)=0.990)


## 4. Online Resources

- [scikit-learn: Naive Bayes user guide](https://scikit-learn.org/stable/modules/naive_bayes.html)
- [scikit-learn: `GaussianNB` API reference](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
- [scikit-learn: `MultinomialNB` API reference](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html)
- [scikit-learn: `BernoulliNB` API reference](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.BernoulliNB.html)
- [scikit-learn: `load_wine` dataset docs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)
